In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

In [ ]:
from google import genai

client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="什麼是AI AGENT？"
)
print(response.text)

In [ ]:
def stateless_query(payload):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=payload
    )
    return response.text


In [ ]:
result = stateless_query("簡介明新科技大學")
print(result)

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError

from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

# =========================
# LINE Bot 設定
# =========================
line_channel_access_token = "你的 Channel Access Token"
line_channel_secret = "你的 Channel Secret"
port = 5000

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


# =========================
# Gemini 回應函式
# =========================
# 這裡假設你已經有寫好 stateless_query()
# 這個函式負責把 prompt 傳給 Gemini，然後回傳 Gemini 的回答
def stateless_query(prompt):
    # 這裡放你的 Gemini API 程式
    # 範例：
    # response = client.models.generate_content(
    #     model="gemini-xxx",
    #     contents=prompt
    # )
    # return response.text

    return "這裡是 Gemini 回應：" + prompt


@app.route("/", methods=["POST"])
def callback():
    # 取得 LINE 傳來的簽章，用來驗證請求是不是 LINE 官方送來的
    signature = request.headers["X-Line-Signature"]

    # 取得 LINE 傳來的訊息內容
    body = request.get_data(as_text=True)
    print("BODY:", body)
    app.logger.info("Request body: " + body)

    # 將訊息交給 LINE WebhookHandler 處理
    try:
        handler.handle(body, signature)

    except InvalidSignatureError:
        app.logger.info(
            "Invalid signature. Please check your channel access token/channel secret."
        )
        abort(400)

    return "OK"


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    # 取得使用者輸入的文字
    text = event.message.text

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # ======================================================
        # 判斷使用者輸入的文字是否以 "AI " 開頭
        #
        # 為什麼要以 "AI " 開頭，Gemini 才會回應？
        #
        # 1. 用來區分「一般訊息」和「AI 問答訊息」
        #    如果沒有加這個判斷，使用者傳任何文字都會丟給 Gemini。
        #
        # 2. 避免 Gemini API 被一直呼叫
        #    每次呼叫 Gemini 都會消耗 API 額度。
        #    如果所有訊息都傳給 Gemini，很容易造成額度用完，
        #    例如出現 429 RESOURCE_EXHAUSTED 錯誤。
        #
        # 3. 讓使用者主動決定什麼時候要使用 AI
        #    使用者輸入「AI 你好」代表要 Gemini 回答。
        #    使用者只輸入「你好」則代表普通訊息，不進入 Gemini 流程。
        #
        # 4. "AI " 後面有一個空格
        #    所以正確格式是：
        #    AI 你好
        #    AI 幫我介紹 Python
        #
        #    如果輸入「AI你好」沒有空格，就不會觸發 Gemini。
        # ======================================================
        if text.startswith("AI "):

            # 將前面的 "AI " 移除，只留下真正要問 Gemini 的內容
            #
            # 例如：
            # 使用者輸入：AI 你好
            # text[3:] 會取得：你好
            #
            # 因為 "AI " 總共是 3 個字元：
            # A = 第 1 個
            # I = 第 2 個
            # 空格 = 第 3 個
            prompt = text[3:].strip()

            # 呼叫 Gemini，讓 Gemini 根據使用者的問題產生回應
            reply_text = stateless_query(prompt)

            # 將 Gemini 的回答回覆給使用者
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=reply_text)
                    ]
                )
            )

        else:
            # ======================================================
            # 如果使用者輸入的文字不是以 "AI " 開頭
            # 就不會呼叫 Gemini
            #
            # 這樣可以避免一般訊息也被送進 Gemini，
            # 減少 API 額度消耗，也讓 Bot 的功能比較清楚。
            #
            # 這裡是普通回覆流程：
            # 使用者輸入什麼，Bot 就回覆什麼。
            #
            # 但因為 messages 裡面放了兩個 TextMessage，
            # 所以 LINE Bot 會回覆兩次相同文字。
            # ======================================================
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)
                    ]
                )
            )


if __name__ == "__main__":
    app.run(port=port)